In [ ]:
from datetime import datetime
import json
from pathlib import Path
from typing import Literal
import torch
from torch import Tensor
from fusiontimeseries.lib.benchmarking import rmse_with_standard_error
from fusiontimeseries.lib.config import FTSConfig
from fusiontimeseries.lib.dataset import FluxData, TimeseriesDataset
import numpy as np

type PerformanceData = dict[str, dict[int, list[float]]]
type BenchmarkData = dict[Literal["ood", "id"], dict[int, FluxData]]


class ZeroshotBenchmarker:
    BENCHMARK_FLUX_TS_LENGTH: int = 266
    CONTEXT_LENGTH_STEP: int = 20

    def __init__(self, fts_config: FTSConfig) -> None:
        self.fts_config = fts_config
        self.benchmark_data: BenchmarkData = (
            TimeseriesDataset.get_benchmark_flux_traces(config=self.fts_config)
        )
        self.train_data: list[FluxData] = TimeseriesDataset.load_flux_data(
            config=self.fts_config
        )
        self.model = None
        self.model_slug: str | None = None
        torch.cuda.empty_cache()

        # set seed for reproducibility
        torch.manual_seed(fts_config.random_seed)
        torch.cuda.manual_seed_all(fts_config.random_seed)
        np.random.seed(fts_config.random_seed)

    @torch.inference_mode()
    def call_model(self, ctx: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError(
            "call_model method not implemented. Please implement this method to call the model for predictions."
        )

    def autoregressive_rollout(
        self, ctx: torch.Tensor, rollout_horizon: int
    ) -> np.ndarray:
        while len(ctx) < rollout_horizon:
            median_forecast = self.call_model(ctx)
            if len(median_forecast) > self.fts_config.prediction_length:
                print(
                    f"Warning: Model returned more predictions ({len(median_forecast)}) than expected ({self.fts_config.prediction_length}). Truncating to expected length."
                )
            ctx = torch.cat((ctx, median_forecast), dim=0)
        return ctx[:rollout_horizon].cpu().numpy()

    def sample_rollout(
        self, start_context_length: int, flux_data: FluxData
    ) -> np.ndarray:
        time_series: np.ndarray = np.array(flux_data.energy_flux)
        ctx = torch.tensor(time_series[:start_context_length], dtype=torch.float32)
        forecast: np.ndarray = self.autoregressive_rollout(
            ctx, rollout_horizon=len(time_series)
        )
        return forecast

    def run(
        self, data: dict[str, dict[int, FluxData]] | None = None
    ) -> PerformanceData:
        if self.model is None:
            raise ValueError(
                "Model not set. Please set the model before running the benchmark."
            )

        max_context_length: int = (
            self.BENCHMARK_FLUX_TS_LENGTH - self.fts_config.prediction_length
        )
        start_context_length: int = max_context_length % self.CONTEXT_LENGTH_STEP

        _data = data or self.benchmark_data
        performance: PerformanceData = {**{namespace: {} for namespace in _data.keys()}}
        for namespace, samples in _data.items():
            print(f"Evaluating Namespace: {namespace} with {len(samples)} samples")
            for context_length in range(
                start_context_length, max_context_length + 1, self.CONTEXT_LENGTH_STEP
            ):
                true_tail_mean: list[float] = []
                pred_tail_mean: list[float] = []
                for _, flux_data in samples.items():
                    forecast: np.ndarray = self.sample_rollout(
                        start_context_length=context_length,
                        flux_data=flux_data,
                    )

                    pred_tail_mean.append(
                        forecast[-self.fts_config.pred_tail_timestamps :].mean()
                    )
                    true_tail_mean.append(
                        float(
                            np.mean(
                                flux_data.energy_flux[
                                    -self.fts_config.pred_tail_timestamps :
                                ],
                                dtype=np.float32,
                            )
                        )
                    )
                rsme, rsme_se = rmse_with_standard_error(
                    np.array(true_tail_mean), np.array(pred_tail_mean)
                )
                performance[namespace][context_length] = [rsme, rsme_se]
        return performance

    def save_results(self, performance: PerformanceData) -> None:
        if self.model_slug is None:
            raise ValueError(
                "Model slug not set. Please set the model slug before saving results."
            )

        model_name_clean = self.model_slug.replace("/", "_")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Save results to JSON
        data_dir = Path(".").resolve() / "results" / model_name_clean
        data_dir.mkdir(parents=True, exist_ok=True)
        results_file = (
            data_dir / f"{timestamp}_{model_name_clean}_zeroshot_performance.json"
        )
        with open(results_file, "w") as f:
            json.dump(performance, f, indent=2)

        print(f"Results saved to: {results_file}")

In [ ]:
fts_config = FTSConfig()
fts_config.prediction_length = 64

In [ ]:
from typing import Any


benchmark_data: dict[str, Any] = TimeseriesDataset.get_benchmark_flux_traces(
    config=fts_config
)  # type: ignore
flux_data: list[FluxData] = TimeseriesDataset.load_flux_data(config=fts_config)
validation_data = [flux_data for flux_data in flux_data if flux_data.is_validation]
print(
    f"Loaded {len(flux_data)} flux data samples, with {len(validation_data)} validation samples."
)
benchmark_data["val"] = {flux_data.idx: flux_data for flux_data in validation_data}

# Chronos 2

In [ ]:
from chronos import Chronos2Pipeline
from fusiontimeseries.lib.config import FTSConfig


class Chronos2Benchmarker(ZeroshotBenchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "amazon/chronos-2"
        self.model = Chronos2Pipeline.from_pretrained(
            pretrained_model_name_or_path=self.model_slug,
            device_map=fts_config.device,
            dtype=torch.bfloat16,
        )
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: torch.Tensor) -> torch.Tensor:
        # quantile_outputs: list(n_variates, n_quantiles, prediction_length)
        quantile_outputs: list[torch.Tensor] = self.model.predict(
            # inputs: (n_series, n_variates, context_length)
            inputs=ctx.unsqueeze(0).unsqueeze(0),
            prediction_length=self.fts_config.prediction_length,
        )
        quantiles: torch.Tensor = quantile_outputs[0]

        median_quantile_idx: int = quantiles.shape[1] // 2
        median_forecast: torch.Tensor = quantiles[0, median_quantile_idx, :].cpu()
        return median_forecast


benchmarker = Chronos2Benchmarker(fts_config=fts_config)
performance = benchmarker.run(benchmark_data)
benchmarker.save_results(performance)

# Chronos Bolt Tiny

In [ ]:
from chronos import ChronosBoltPipeline


class ChronosBoltBenchmarker(Chronos2Benchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "amazon/chronos-bolt-tiny"
        self.model = ChronosBoltPipeline.from_pretrained(
            pretrained_model_name_or_path=self.model_slug,
            device_map=fts_config.device,
            dtype=torch.bfloat16,
        )
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        # quantiles: (batch_size, num_quantiles, prediction_length)
        quantiles: torch.Tensor = self.model.predict(
            inputs=ctx.unsqueeze(0).to(fts_config.device),
            prediction_length=fts_config.prediction_length,
        )

        median_quantile_idx: int = quantiles.shape[1] // 2
        median_forecast: torch.Tensor = quantiles[0, median_quantile_idx, :].cpu()
        return median_forecast


bolt_benchmarker = ChronosBoltBenchmarker(fts_config=fts_config)
bolt_performance = bolt_benchmarker.run(benchmark_data)
bolt_benchmarker.save_results(bolt_performance)
bolt_performance

# TimesFM2.0-500m

In [ ]:
!uv remove "timesfm[torch] @ git+https://github.com/google-research/timesfm.git"
!uv add timesfm

In [ ]:
from timesfm.timesfm_torch import TimesFmTorch
from timesfm.timesfm_base import TimesFmHparams, TimesFmCheckpoint


class TimesFM2p0500MBenchmarker(ZeroshotBenchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "google/timesfm-2.0-500m-pytorch"
        hparams = TimesFmHparams(
            backend="gpu",
            per_core_batch_size=fts_config.batch_size,
            horizon_len=fts_config.prediction_length,
            context_len=fts_config.context_length,
            num_layers=50,
            use_positional_embedding=True,
        )
        self.model = TimesFmTorch(
            hparams=hparams,
            checkpoint=TimesFmCheckpoint(huggingface_repo_id=self.model_slug),
        )
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        point_forecast, _ = self.model.forecast(
            inputs=[ctx],
            freq=[1],
            forecast_context_len=fts_config.context_length,
            normalize=True,
        )
        return torch.from_numpy(point_forecast).cpu().squeeze(0)


timesfm_benchmarker = TimesFM2p0500MBenchmarker(fts_config=fts_config)
# timesfm_performance = timesfm_benchmarker.run(benchmark_data)
# timesfm_benchmarker.save_results(timesfm_performance)
# timesfm_performance

# TimesFM2.5-200M

In [ ]:
!uv remove timesfm
!uv add "timesfm[torch] @ git+https://github.com/google-research/timesfm.git"

In [ ]:
import timesfm


class TimesFM2p5200MBenchmarker(ZeroshotBenchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "google/timesfm-2.5-200m-pytorch"
        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
            self.model_slug, torch_compile=True
        )
        self.model.compile(
            timesfm.ForecastConfig(
                max_context=fts_config.context_length,
                per_core_batch_size=1,
                max_horizon=fts_config.prediction_length,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            )
        )
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        forecast, _ = self.model.forecast(
            inputs=[ctx],  # type: ignore
            horizon=fts_config.prediction_length,
        )
        return torch.from_numpy(forecast).cpu().squeeze(0)


timesfm_2p5_benchmarker = TimesFM2p5200MBenchmarker(fts_config=fts_config)
timesfm_2p5_performance = timesfm_2p5_benchmarker.run(benchmark_data)
timesfm_2p5_benchmarker.save_results(timesfm_2p5_performance)
timesfm_2p5_performance

# TiRex

In [ ]:
from tirex import load_model, ForecastModel  # noqa: E402


class TiRexBenchmarker(ZeroshotBenchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "NX-AI/TiRex"
        self.model: ForecastModel = load_model(
            path=self.model_slug, device=fts_config.device, backend="torch"
        )  # type: ignore
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        # quantile_outputs: (batch_size, prediction_length, num_quantiles)
        quantiles, _ = self.model.forecast(
            context=ctx.to(fts_config.device),
            prediction_length=fts_config.prediction_length,
        )  # type: ignore
        quantiles: torch.Tensor

        median_quantile_idx: int = quantiles.shape[-1] // 2
        median_forecast: torch.Tensor = quantiles[0, :, median_quantile_idx].cpu()
        return median_forecast


tirex_benchmarker = TiRexBenchmarker(fts_config=fts_config)
tirex_performance = tirex_benchmarker.run(benchmark_data)
tirex_benchmarker.save_results(tirex_performance)
tirex_performance

# Plot

In [ ]:
# gather all json files with *zeroshot_performance.json suffix in results directory
# for each model folder, take only the newest file
results_dir = Path(".").resolve() / "results"
performance_files = []

for model_dir in results_dir.iterdir():
    if model_dir.is_dir():
        model_perf_files = list(model_dir.glob("*zeroshot_performance.json"))
        if model_perf_files:
            # Get the newest file based on filename timestamp (or modification time)
            newest_file = max(model_perf_files, key=lambda p: p.stat().st_mtime)
            performance_files.append(newest_file)

performance_files

In [ ]:
# load this file as well: C:\Users\sever\code\academics\master\fusiontimeseries\src\fusiontimeseries\ablations\results\lora-ablation-tailmeansampling-0\cl_rmse_results.json
timesfm_lora_performance_file = (
    Path(".").resolve().parent
    / "ablations"
    / "results"
    / "lora-ablation-tailmeansampling-0"
    / "cl_rmse_results.json"
)
with open(timesfm_lora_performance_file, "r") as f:
    timesfm_lora_performance = json.load(f)

refactored_timesfm_lora_performance = {
    "id": {
        entry["context_length"]: [entry["rmse"], entry["rmse_standard_error"]]
        for entry in timesfm_lora_performance
    }
}
refactored_timesfm_lora_performance

In [ ]:
# read all performance data into a dictionary
all_performance_data: dict[str, PerformanceData] = {}
for perf_file in performance_files:
    with open(perf_file, "r") as f:
        performance_data = json.load(f)
        all_performance_data[perf_file.parent.stem + perf_file.parent.suffix] = (
            performance_data
        )

In [ ]:
all_performance_data["timesfm-2.0-500M-lora"] = refactored_timesfm_lora_performance

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

markers = ["o", "s", "^", "D", "v"]

for idx, (model_name, performance_data) in enumerate(all_performance_data.items()):
    id_performance = performance_data["id"]
    context_lengths = sorted(id_performance.keys(), key=lambda x: int(x), reverse=False)
    rmses = [id_performance[cl][0] for cl in context_lengths]
    rmse_ses = [id_performance[cl][1] for cl in context_lengths]
    plt.errorbar(
        [int(c) + (idx - 2) * 2 for c in context_lengths],
        rmses,
        yerr=rmse_ses,
        label=model_name,
        marker=markers[idx % len(markers)],
        capsize=2,
        alpha=0.7,
    )
    # plt.plot(context_lengths, rmses, label=model_name, marker=markers[idx % len(markers)])
# plt.xlabel("Context Length")
plt.xticks([int(c) for c in range(2, 203, 20)])
plt.xlabel("Start Context Length (with jitter for visibility)")
plt.ylabel("RMSE ± Standard Error")
plt.title("Zero-Shot and Finetuned Performance on ID Energy Flux")
plt.legend()
plt.grid(axis="y")
plt.show()

In [ ]:
# plot all ood performance data in a single plot
plt.figure(figsize=(12, 6))
for idx, (model_name, performance_data) in enumerate(all_performance_data.items()):
    ood_performance = performance_data["ood"]
    context_lengths = sorted(
        ood_performance.keys(), key=lambda x: int(x), reverse=False
    )
    rmses = [ood_performance[cl][0] for cl in context_lengths]
    rmse_ses = [ood_performance[cl][1] for cl in context_lengths]
    plt.errorbar(
        [int(c) + (idx - 2) * 2 for c in context_lengths],
        rmses,
        yerr=rmse_ses,
        label=model_name,
        marker=markers[idx % len(markers)],
        capsize=2,
        alpha=0.7,
    )
    # plt.plot(context_lengths, rmses, label=model_name, marker=markers[idx % len(markers)])
# plt.xlabel("Context Length")
plt.xticks([int(c) for c in range(2, 203, 20)])
plt.xlabel("Start Context Length (with jitter for visibility)")
plt.ylabel("RMSE ± Standard Error")
plt.title("Zero-Shot Performance on OOD Energy Flux")
plt.legend()
plt.grid(axis="y")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

markers = ["o", "s", "^", "D", "v"]

for idx, (model_name, performance_data) in enumerate(all_performance_data.items()):
    id_performance = performance_data["val"]
    context_lengths = sorted(id_performance.keys(), key=lambda x: int(x), reverse=False)
    rmses = [id_performance[cl][0] for cl in context_lengths]
    rmse_ses = [id_performance[cl][1] for cl in context_lengths]
    plt.errorbar(
        [int(c) + (idx - 2) * 2 for c in context_lengths],
        rmses,
        yerr=rmse_ses,
        label=model_name,
        marker=markers[idx % len(markers)],
        capsize=2,
        alpha=0.7,
    )
    # plt.plot(context_lengths, rmses, label=model_name, marker=markers[idx % len(markers)])
# plt.xlabel("Context Length")
plt.xticks([int(c) for c in range(2, 203, 20)])
plt.xlabel("Start Context Length (with jitter for visibility)")
plt.ylabel("RMSE ± Standard Error")
plt.yticks(range(0, 120, 20))
plt.ylim(0, 120)
plt.title("Zero-Shot Performance on Validation Energy Flux")
# plt.legend()
plt.grid(axis="y")
plt.show()

# Inspection

TimesFM2.0-500M has suspicious outlier in id distribution for context 42, 62, and 82.

In [ ]:
namespace = "val"
context_lengths = [82, 102]

In [ ]:
forecasts: dict[int, dict[int, np.ndarray]] = {**{cl: {} for cl in context_lengths}}
for context_length in context_lengths:
    for _, flux_data in benchmark_data[namespace].items():
        forecast: np.ndarray = timesfm_benchmarker.sample_rollout(
            start_context_length=context_length,
            flux_data=flux_data,
        )
        forecasts[context_length][flux_data.idx] = forecast

In [ ]:
# For each context length, plot the true vs predicted trajectory of the given flux data sample
import matplotlib.pyplot as plt

for context_length in context_lengths:
    num_samples = len(forecasts[context_length])
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 4 * num_samples))

    # Handle case when there's only one sample
    if num_samples == 1:
        axes = [axes]

    for idx, (flux_data_id, forecast) in enumerate(forecasts[context_length].items()):
        true_time_series: np.ndarray = np.array(
            benchmark_data[namespace][flux_data_id].energy_flux
        )

        axes[idx].plot(
            true_time_series, label=f"True Flux {flux_data_id}", alpha=0.7, linewidth=2
        )
        axes[idx].plot(
            forecast,
            label=f"Predicted Flux {flux_data_id}",
            alpha=0.7,
            linewidth=2,
            linestyle="--",
        )
        axes[idx].axvline(
            x=context_length, color="red", linestyle=":", alpha=0.5, label="Context End"
        )
        axes[idx].set_xlabel("Time Step")
        axes[idx].set_ylabel("Energy Flux")
        axes[idx].set_title(f"Sample ID {flux_data_id}")
        axes[idx].legend()
        axes[idx].grid(alpha=0.3)

    fig.suptitle(
        f"True vs Predicted Energy Flux Trajectories (Context Length: {context_length})",
        fontsize=14,
        y=1.001,
    )
    plt.tight_layout()
    plt.show()

We evaluated the zero-shot performance on a held out test set of 6 in-distribution (ID) and 5 out-of-distribution (OOD) flux trajectories, as well as a pre-defined validation set of 3 flux traces. To assess how increasing initial context length affects the overall RMSE on the mean tail flux trajectory prediction we report RMSE and standard error for all models on different starting context lengths. For ID and OOD data all models were within standard error margins except timesfm-2.0-500M, which showed a surprising performance for starting context lengths 42, 62, and 82. On the validation trajectories, however, timesfm-2.0-500M was the only model which showed severe performance outliers for starting context lengths 102, 142, and 202.

For our ablation studies we nevertheless chose timesfm-2.0-500M as the base foundation model:
- In the downsampled trajectory regime where a flux series only consists of 266 timesteps, looking beyond the first 80 timesteps can expose regions where the turbulent flux already reaches the saturated zone. So, we are interested in the zero-shot performance with context-lengths up to the first 80 timesteps.
- In all evaluations on ID, OOD and validation data all models performed within standard error margins in the interesting initial context range. TimesFM-2.0-500M was the only one showing superior performance on ID data within this context length range.

# Normalization Ablations

In [ ]:
from timesfm import pytorch_patched_decoder as timesfm_lib

fts_config.prediction_length = 128


def _masked_mean_std(
    inputs: torch.Tensor, padding: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Calculates mean and standard deviation of `inputs` across axis 1.

    It excludes values where `padding` is 1.

    Args:
      inputs: A PyTorch tensor of shape [b, n, p].
      padding: A PyTorch tensor of shape [b, n, p] with values 0 or 1.

    Returns:
      A tuple containing the mean and standard deviation.
      We return the statistics of the first patch with more than three non-padded
      values.
    """
    arr = inputs
    pad = padding

    # Create a mask where padding is 0
    mask = 1 - pad

    masked_median = torch.median(arr[mask == 1])
    masked_median = masked_median.unsqueeze(0)
    masked_iqr = torch.quantile(arr[mask == 1], 0.75) - torch.quantile(
        arr[mask == 1], 0.25
    )
    masked_iqr = masked_iqr.unsqueeze(0)

    # masked_mean = torch.mean(arr[mask == 1])
    # masked_mean = masked_mean.unsqueeze(0)
    # masked_std = torch.std(arr[mask == 1])
    # masked_std = masked_std.unsqueeze(0)

    # Calculate the number of valid elements
    # num_valid_elements = torch.sum(mask, dim=1)
    # num_valid_elements = torch.where(
    #     num_valid_elements == 0,
    #     torch.tensor(1,
    #                  dtype=num_valid_elements.dtype,
    #                  device=num_valid_elements.device),
    #     num_valid_elements,
    # )

    # # Calculate the masked sum and squared sum
    # masked_sum = torch.sum(arr * mask, dim=1)
    # masked_squared_sum = torch.sum((arr * mask)**2, dim=1)

    # # Calculate the masked mean and standard deviation
    # masked_mean = masked_sum / num_valid_elements
    # masked_var = masked_squared_sum / num_valid_elements - masked_mean**2
    # masked_var = torch.where(
    #     masked_var < 0.0,
    #     torch.tensor(0.0, dtype=masked_var.dtype, device=masked_var.device),
    #     masked_var,
    # )
    # masked_std = torch.sqrt(masked_var)

    return masked_median, masked_iqr
    # return masked_mean, masked_std


timesfm_lib._masked_mean_std = _masked_mean_std

In [ ]:
class TimesFM2p0500MBenchmarker(ZeroshotBenchmarker):
    def __init__(self, fts_config: FTSConfig) -> None:
        super().__init__(fts_config)
        self.model_slug = "google/timesfm-2.0-500m-pytorch"
        hparams = TimesFmHparams(
            backend="gpu",
            per_core_batch_size=fts_config.batch_size,
            horizon_len=fts_config.prediction_length,
            context_len=fts_config.context_length,
            num_layers=50,
            use_positional_embedding=True,
        )
        self.model = TimesFmTorch(
            hparams=hparams,
            checkpoint=TimesFmCheckpoint(huggingface_repo_id=self.model_slug),
        )
        print(
            f"Model {self.model_slug} loaded successfully on device {fts_config.device}."
        )

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        point_forecast, _ = self.model.forecast(
            inputs=[ctx],
            freq=[1],
            forecast_context_len=fts_config.context_length,
            normalize=False,
        )
        return torch.from_numpy(point_forecast).cpu().squeeze(0)


timesfm_benchmarker = TimesFM2p0500MBenchmarker(fts_config=fts_config)
timesfm_performance = timesfm_benchmarker.run(benchmark_data)
# timesfm_benchmarker.save_results(timesfm_performance)
timesfm_performance

In [ ]:
timesfm_performance_wiht_new_stats = timesfm_performance.copy()

In [ ]:
timesfm_benchmarker.save_results(timesfm_performance_wiht_new_stats)